In [10]:
import pandas as pd
import re
import ast
from sentence_transformers import SentenceTransformer, util
from langchain_groq import ChatGroq
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [3]:
df=pd.read_csv(r"C:\Users\asuna\Downloads\final_address_variants.csv")

In [4]:
#creating a list of parent and variant addresses
parent_addrs = df['Parent_Addresses'].dropna().unique().tolist()
variant_addrs = df['Variant_Addresses'].dropna().unique().tolist()

In [5]:
#normalizing the parent addresses
def normalize_parent(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text) 
    return text.strip()

#normalizing the variant addresses
def normalize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # remove punctuation
    text = re.sub(r'\s+', ' ', text)  # normalize extra spaces

    # expand common address abbreviations (India-specific)
    replacements = {
        " st ": " street "," rd ": " road "," rd.": " road "," ave ": " avenue "," blvd ": " boulevard "," ln ": " lane "," dr ": " drive "," hsg ": " housing ",
        " soc ": " society ",
        " apt ": " apartment ",
        " appt ": " apartment ",
        " bldg ": " building ",
        " bld ": " building ",
        " nr ": " near ",
        " opp ": " opposite ",
        " nxt ": " next ",
        " ph ": " phase ",
        " rd ":" road ",
        " sec ": " sector ",
        " stn ": " station ",
        " dist ": " district ",
        " tal ": " taluka ",
        " po ": " post office ",
        " ps ": " police station ",
        " ngr ": " nagar ",
        " extn ": " extension ",
        " indl ": " industrial ",
        " ind ": " industrial ",
        " hwy ": " highway ",
        " jn ": " junction ",
        " ml ": " mall ",
        " cplx ": " complex ",
        " clny ": " colony ",
        " cln ": " colony ",
        " qtrs ": " quarters ",
        " qtr ": " quarter ",
        " flr ": " floor ",
        " no ": " number ",
        " pvt ": " private ",
        " ltd ": " limited ",
        " co ": " company ",
        " comp ": " company ",
        " ap ": " andhra pradesh ",
        " tn ": " tamil nadu ",
        " mh ": " maharashtra ",
        " gj ": " gujarat ",
        " ka ": " karnataka ",
        " up ": " uttar pradesh ",
        " mp ": " madhya pradesh ",
        " wb ": " west bengal ",
        " dl ": " delhi ",
        "resi":"residency"
    }

    # apply replacements safely
    for abbr, full in replacements.items():
        text = re.sub(rf"\b{abbr.strip()}\b", full.strip(), text)

    return text.strip()

In [6]:
#normalizing the parent and variant addresses
parent_addrs_clean = [normalize_parent(x) for x in parent_addrs]
variant_addrs_clean = [normalize(x) for x in variant_addrs]

In [7]:
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.2)

In [8]:
prompt_template = ChatPromptTemplate.from_template(
    """You are an expert address matching system. Your task is to map variant addresses to their corresponding parent addresses based on semantic similarity, despite typos, abbreviations, or formatting differences.

**INSTRUCTIONS:**
1. Analyze each variant address carefully
2. Compare it against ALL parent addresses
3. Match each variant to the MOST SIMILAR parent address
4. Consider common variations:
   - Abbreviations (St/Street, Ave/Avenue, Apt/Apartment)
   - Typos and misspellings
   - Different word orders
   - Missing or extra punctuation
   - Case differences
5. If a variant has NO reasonable match, use null as the value

**OUTPUT REQUIREMENTS:**
- Return ONLY a valid JSON object with no exceptions
- No nulls, if a variant address is not match to any of the parent address return the most probable parent address
- No additional text, explanations, or markdown
- Use the exact variant address strings as keys
- Use the exact parent address strings as values
- Format: {{"variant_address": "matched_parent_address"}}

**VARIANT ADDRESSES TO MATCH:**
{variants}

**PARENT ADDRESS OPTIONS:**
{parents}

**OUTPUT (JSON only):**"""
)

prompt = prompt_template.format(
        variants= variant_addrs,
        parents=parent_addrs)

In [12]:
response = llm.invoke(prompt)
llm_output = response.content.strip()

In [13]:
clean_output = llm_output.replace('\\n', '\n')
print(clean_output)

{"Flat 203 sunrise apts jp nagar bangalore": "Flat 203, Sunrise Apartments, JP Nagar, Bengaluru, Karnataka 560078", 
"203 sunrise apartment jp ngr blr": "Flat 203, Sunrise Apartments, JP Nagar, Bengaluru, Karnataka 560078", 
"flat no 203 jp nagar blore": "Flat 203, Sunrise Apartments, JP Nagar, Bengaluru, Karnataka 560078", 
"sunrise apt flat 203 bengaluru": "Flat 203, Sunrise Apartments, JP Nagar, Bengaluru, Karnataka 560078", 
"203 sunrise apmt jp nagr karnataka": "Flat 203, Sunrise Apartments, JP Nagar, Bengaluru, Karnataka 560078", 
"house no 45 sec 21c faridabad": "House 45, Sector 21C, Faridabad, Haryana 121001", 
"45 sector 21 c fbd": "House 45, Sector 21C, Faridabad, Haryana 121001", 
"house forty five 21c haryana": "House 45, Sector 21C, Faridabad, Haryana 121001", 
"sector 21c house45 fbd": "House 45, Sector 21C, Faridabad, Haryana 121001", 
"45 sec21 c faridabad haryana": "House 45, Sector 21C, Faridabad, Haryana 121001", 
"flat5b green meadows andheri w mumbai": "Flat 5B, G

In [14]:
data_dict = ast.literal_eval(clean_output)

In [15]:
type(clean_output)

str

In [16]:
try:
    llm_mapping = ast.literal_eval(clean_output)
    if not isinstance(llm_mapping, dict):
        raise ValueError("LLM did not return a dictionary.")
except Exception as e:
    print(f"⚠️ Could not parse LLM output properly:\n{clean_output}\nError: {e}")
    llm_mapping = {}

In [21]:


# Example: data_dict = {...}  # your dictionary
df_dict = pd.DataFrame.from_dict(llm_mapping, orient='index', columns=['Matched_Parent']).reset_index()
df_dict.rename(columns={'index': 'Variant_Addresses'}, inplace=True)




In [22]:
df_dict

,Variant_Addresses,Matched_Parent
0,Flat 203 sunrise apts jp nagar bangalore,"Flat 203, Sunrise Apartments, JP Nagar, Bengal..."
1,203 sunrise apartment jp ngr blr,"Flat 203, Sunrise Apartments, JP Nagar, Bengal..."
2,flat no 203 jp nagar blore,"Flat 203, Sunrise Apartments, JP Nagar, Bengal..."
3,sunrise apt flat 203 bengaluru,"Flat 203, Sunrise Apartments, JP Nagar, Bengal..."
4,203 sunrise apmt jp nagr karnataka,"Flat 203, Sunrise Apartments, JP Nagar, Bengal..."
...,...,...
178,20 mird jp,"20 MI Road, Jaipur, Rajasthan 302001"
179,mi road 20 jaipur,"20 MI Road, Jaipur, Rajasthan 302001"
180,plot 210 hitech city hyd,"Plot 210, HITEC City, Hyderabad, Telangana 500081"
181,plot no 210 hitec hyderabad,"Plot 210, HITEC City, Hyderabad, Telangana 500081"


In [23]:
comparison = pd.merge(df, df_dict, on="Variant_Addresses", how="left")

# Check where prediction matches the true parent address
comparison['Correct'] = comparison['Parent_Addresses'] == comparison['Matched_Parent']

# Calculate accuracy
accuracy = comparison['Correct'].mean() * 100
accuracy

100.0